In [1]:
import os, gc, json, math, random, hashlib, warnings, time
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score
)

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def seed_everything(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True  # faster

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


In [3]:
@dataclass
class CFG:
    # Data
    data_path: str = "belka_agg_downsampled.parquet"
    out_dir: str = "./ablation_q1_aligned"

    smiles_col: str = "molecule_smiles"
    bb_cols: Tuple[str, str, str] = (
        "buildingblock1_smiles","buildingblock2_smiles","buildingblock3_smiles"
    )
    targets: Tuple[str, str, str] = ("BRD4","HSA","sEH")

    # Split (same logic as Hybrid-Final; expose fold choice)
    split_mode: str = "bb_triplet"   # "bb_triplet" or "random"
    n_splits: int = 5
    test_fold: int = 0
    val_fold: int = 0
    test_frac: float = 0.15
    val_frac: float = 0.15

    # Tokenization / encoder
    base_model: str = "seyonec/ChemBERTa-zinc-base-v1"
    max_len: int = 128

    # Fingerprints
    fp_bits: int = 2048
    fp_radius: int = 2

    # MLM / DAPT (runtime-aware but faithful)
    do_mlm: bool = True
    mlm_out_name: str = "ckpt_mlm_belka"
    mlm_sample_max: int = 250_000     # reduce if needed
    mlm_val_max: int = 50_000
    mlm_max_steps: int = 5_000        # fixed-step DAPT for speed
    mlm_batch: int = 32
    mlm_lr: float = 5e-5
    mlm_prob: float = 0.15

    # Supervised training (faithful structure; capped compute)
    batch_size: int = 32
    eval_batch_mult: int = 2
    use_fp16: bool = True

    # “Paper-faithful” objective
    lambda_promiscuity: float = 0.05
    target_precision: float = 0.90

    # Two-phase schedule (cap by steps to control runtime)
    epochs_head: int = 1
    epochs_finetune: int = 3          # set to 3 to match Hybrid-Final exactly
    max_steps_head: Optional[int] = 1200
    max_steps_ft: Optional[int] = 1200
    unfreeze_k: int = 2

    lr_head: float = 2e-3
    lr_encoder: float = 3e-5
    weight_decay: float = 0.01
    warmup_frac: float = 0.08

    # Ablation reporting
    seeds: Tuple[int, ...] = (42,)     # set (42, 52, 62) for mean±std

    # Optional “budget eval” (kept OFF by default)
    eval_max_n: Optional[int] = None   # e.g., 200_000 for quick smoke tests only

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


CFG(data_path='belka_agg_downsampled.parquet', out_dir='./ablation_q1_aligned', smiles_col='molecule_smiles', bb_cols=('buildingblock1_smiles', 'buildingblock2_smiles', 'buildingblock3_smiles'), targets=('BRD4', 'HSA', 'sEH'), split_mode='bb_triplet', n_splits=5, test_fold=0, val_fold=0, test_frac=0.15, val_frac=0.15, base_model='seyonec/ChemBERTa-zinc-base-v1', max_len=128, fp_bits=2048, fp_radius=2, do_mlm=True, mlm_out_name='ckpt_mlm_belka', mlm_sample_max=250000, mlm_val_max=50000, mlm_max_steps=5000, mlm_batch=32, mlm_lr=5e-05, mlm_prob=0.15, batch_size=32, eval_batch_mult=2, use_fp16=True, lambda_promiscuity=0.05, target_precision=0.9, epochs_head=1, epochs_finetune=3, max_steps_head=1200, max_steps_ft=1200, unfreeze_k=2, lr_head=0.002, lr_encoder=3e-05, weight_decay=0.01, warmup_frac=0.08, seeds=(42,), eval_max_n=None)


In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

canon_cache = os.path.join(cfg.out_dir, "canonicalized.parquet")

def canon_smiles(s: str) -> str:
    try:
        m = Chem.MolFromSmiles(str(s))
        if m is None:
            return ""
        return Chem.MolToSmiles(m, isomericSmiles=True)
    except Exception:
        return ""

if os.path.exists(canon_cache):
    df = pd.read_parquet(canon_cache)
    print("[cache] loaded:", canon_cache, df.shape)
else:
    df = pd.read_parquet(cfg.data_path)
    print("raw:", df.shape)

    # Canonicalize molecule + building blocks (Hybrid-Final faithful)
    tqdm.pandas()
    df[cfg.smiles_col] = df[cfg.smiles_col].astype(str).progress_map(canon_smiles)
    for c in cfg.bb_cols:
        df[c] = df[c].astype(str).progress_map(canon_smiles)

    before = len(df)
    df = df[df[cfg.smiles_col].str.len() > 0].reset_index(drop=True)
    print("dropped invalid molecule smiles:", before - len(df), "remaining:", len(df))

    df.to_parquet(canon_cache, index=False)
    print("[cache] saved:", canon_cache)

# Labels
for t in cfg.targets:
    df[t] = pd.to_numeric(df[t], errors="coerce").fillna(0).astype(np.int8)

df["y_combo"] = df[list(cfg.targets)].astype(str).agg("".join, axis=1)
df["y_sum"] = df[list(cfg.targets)].sum(axis=1).astype(np.int16)

# Leakage-safe group
df["bb_triplet"] = df[cfg.bb_cols[0]] + "||" + df[cfg.bb_cols[1]] + "||" + df[cfg.bb_cols[2]]

print("final df:", df.shape)
print(df["y_combo"].value_counts().head(10))


raw: (1609779, 8)


  0%|          | 0/1609779 [00:00<?, ?it/s]

  0%|          | 0/1609779 [00:00<?, ?it/s]

  0%|          | 0/1609779 [00:00<?, ?it/s]

  0%|          | 0/1609779 [00:00<?, ?it/s]

dropped invalid molecule smiles: 0 remaining: 1609779
[cache] saved: ./ablation_q1_aligned\canonicalized.parquet
final df: (1609779, 11)
y_combo
001    644702
100    456070
010    328942
000    100000
011     79171
101       597
110       235
111        62
Name: count, dtype: int64


In [5]:
def make_split(df_: pd.DataFrame):
    if cfg.split_mode == "random":
        tr, temp = train_test_split(
            df_, test_size=cfg.test_frac + cfg.val_frac,
            stratify=df_["y_combo"], random_state=42
        )
        rel = cfg.val_frac / (cfg.test_frac + cfg.val_frac)
        va, te = train_test_split(temp, test_size=1-rel, stratify=temp["y_combo"], random_state=43)
        return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

    sgkf = StratifiedGroupKFold(n_splits=cfg.n_splits, shuffle=True, random_state=42)
    folds = list(sgkf.split(df_, df_["y_combo"], groups=df_["bb_triplet"]))
    trval_idx, te_idx = folds[cfg.test_fold]
    trval = df_.iloc[trval_idx].copy()
    te = df_.iloc[te_idx].copy()

    sgkf2 = StratifiedGroupKFold(n_splits=cfg.n_splits, shuffle=True, random_state=43)
    folds2 = list(sgkf2.split(trval, trval["y_combo"], groups=trval["bb_triplet"]))
    tr_idx, va_idx = folds2[cfg.val_fold]
    tr = trval.iloc[tr_idx].copy()
    va = trval.iloc[va_idx].copy()

    overlap = len(set(tr["bb_triplet"]).intersection(set(te["bb_triplet"])))
    print("Group overlap(train,test) =", overlap, "(should be 0)")
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

train_df, val_df, test_df = make_split(df)
print("sizes:", len(train_df), len(val_df), len(test_df))

# Add stable indices so we can safely subset df while using full fp/bb arrays
train_df["split_idx"] = np.arange(len(train_df), dtype=np.int64)
val_df["split_idx"]   = np.arange(len(val_df), dtype=np.int64)
test_df["split_idx"]  = np.arange(len(test_df), dtype=np.int64)

y_tr = train_df[list(cfg.targets)].values.astype(np.int64)
y_va = val_df[list(cfg.targets)].values.astype(np.int64)
y_te = test_df[list(cfg.targets)].values.astype(np.int64)


Group overlap(train,test) = 0 (should be 0)
sizes: 1030258 257565 321956


In [6]:
def build_bb_vocab(train_df_: pd.DataFrame) -> Dict[str, int]:
    bbs = pd.concat([train_df_[c].astype(str) for c in cfg.bb_cols]).unique().tolist()
    vocab = {"[PAD]": 0, "[UNK]": 1}
    for b in bbs:
        if b not in vocab:
            vocab[b] = len(vocab)
    return vocab

bb_vocab = build_bb_vocab(train_df)
bb_pad = bb_vocab["[PAD]"]
bb_unk = bb_vocab["[UNK]"]
print("bb_vocab size:", len(bb_vocab))

def encode_bb(df_: pd.DataFrame) -> np.ndarray:
    arr = np.zeros((len(df_), 3), dtype=np.int64)
    for i,(b1,b2,b3) in enumerate(zip(df_[cfg.bb_cols[0]], df_[cfg.bb_cols[1]], df_[cfg.bb_cols[2]])):
        arr[i,0] = bb_vocab.get(str(b1), bb_unk)
        arr[i,1] = bb_vocab.get(str(b2), bb_unk)
        arr[i,2] = bb_vocab.get(str(b3), bb_unk)
    return arr

bb_tr = encode_bb(train_df)
bb_va = encode_bb(val_df)
bb_te = encode_bb(test_df)


bb_vocab size: 1147


In [7]:
fp_dir = os.path.join(cfg.out_dir, "fp_cache")
os.makedirs(fp_dir, exist_ok=True)

def split_signature(df_split: pd.DataFrame, n: int = 20000) -> str:
    cols = [cfg.smiles_col, "bb_triplet"]
    s = df_split[cols].astype(str).head(n).agg("||".join, axis=1).str.cat(sep="##")
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:12]

def fp_memmap_path(split_name: str, df_split: pd.DataFrame) -> str:
    sig = split_signature(df_split)
    return os.path.join(fp_dir, f"{split_name}_fp_packed_{cfg.fp_bits}b_{sig}.npy")

def fp_to_packed(smiles: str) -> np.ndarray:
    m = Chem.MolFromSmiles(smiles)
    bits = np.zeros(cfg.fp_bits, dtype=np.uint8)
    if m is not None:
        bv = AllChem.GetMorganFingerprintAsBitVect(m, cfg.fp_radius, nBits=cfg.fp_bits)
        DataStructs.ConvertToNumpyArray(bv, bits)
    return np.packbits(bits)

def build_fp_memmap(df_split: pd.DataFrame, split_name: str) -> str:
    path = fp_memmap_path(split_name, df_split)
    if os.path.exists(path):
        print("[fp cache] exists:", path)
        return path

    n = len(df_split)
    pack_len = (cfg.fp_bits + 7) // 8
    mm = np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8, shape=(n, pack_len))
    print(f"[fp] building {split_name} n={n} -> {path}")

    for i, smi in enumerate(tqdm(df_split[cfg.smiles_col].astype(str).values, desc=f"FP[{split_name}]")):
        mm[i] = fp_to_packed(smi)
        if (i + 1) % 200_000 == 0:
            mm.flush()

    del mm
    return path

fp_tr_path = build_fp_memmap(train_df, "train")
fp_va_path = build_fp_memmap(val_df,   "val")
fp_te_path = build_fp_memmap(test_df,  "test")

fp_tr = np.load(fp_tr_path, mmap_mode="r")
fp_va = np.load(fp_va_path, mmap_mode="r")
fp_te = np.load(fp_te_path, mmap_mode="r")

print("fp shapes (packed):", fp_tr.shape, fp_va.shape, fp_te.shape)


[fp] building train n=1030258 -> ./ablation_q1_aligned\fp_cache\train_fp_packed_2048b_c4531d84315c.npy


FP[train]:   0%|          | 0/1030258 [00:00<?, ?it/s]

[fp] building val n=257565 -> ./ablation_q1_aligned\fp_cache\val_fp_packed_2048b_3f748105763f.npy


FP[val]:   0%|          | 0/257565 [00:00<?, ?it/s]

[fp] building test n=321956 -> ./ablation_q1_aligned\fp_cache\test_fp_packed_2048b_3c8a50971f8a.npy


FP[test]:   0%|          | 0/321956 [00:00<?, ?it/s]

fp shapes (packed): (1030258, 256) (257565, 256) (321956, 256)


In [8]:
tok_base = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)

class BelkaDataset(Dataset):
    def __init__(self, df_split, fp_mm, bb_arr, y: Optional[np.ndarray]):
        self.df = df_split.reset_index(drop=True)
        self.fp = fp_mm
        self.bb = bb_arr
        self.y = None if y is None else y.astype(np.float32)

        self.smiles = self.df[cfg.smiles_col].astype(str).tolist()
        self.map_idx = self.df["split_idx"].astype(np.int64).to_numpy()

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        rid = int(self.map_idx[i])
        item = {
            "smiles": self.smiles[i],
            "fp_packed": self.fp[rid],
            "bb_ids": self.bb[rid],
        }
        if self.y is not None:
            item["labels"] = self.y[i]
        return item

def make_collate(tokenizer, use_fp: bool, use_bb: bool):
    def _collate(batch):
        smiles = [b["smiles"] for b in batch]
        enc = tokenizer(
            smiles, padding=True, truncation=True, max_length=cfg.max_len, return_tensors="pt"
        )

        fp_packed = np.stack([b["fp_packed"] for b in batch])  # (B, 256)
        fp_bits = np.unpackbits(fp_packed, axis=1)[:, :cfg.fp_bits].astype(np.float32)
        if not use_fp:
            fp_bits[:] = 0.0
        fp_bits = torch.tensor(fp_bits, dtype=torch.float32)

        bb_ids = torch.tensor(np.stack([b["bb_ids"] for b in batch]), dtype=torch.long)
        if not use_bb:
            # We'll ignore bb inside the model (set to None), avoiding “PAD embedding leakage”.
            bb_ids = None

        out = {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "fp_bits": fp_bits,
            "bb_ids": bb_ids,
        }
        if "labels" in batch[0]:
            out["labels"] = torch.tensor(np.stack([b["labels"] for b in batch]), dtype=torch.float32)
        return out
    return _collate

def maybe_cap_eval(df_split: pd.DataFrame) -> pd.DataFrame:
    if cfg.eval_max_n is None or len(df_split) <= cfg.eval_max_n:
        return df_split
    # IMPORTANT: random cap, NOT label-conditional (no prevalence distortion)
    return df_split.sample(cfg.eval_max_n, random_state=42).reset_index(drop=True)


In [9]:
class MeanAttnPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn = nn.Linear(hidden, 1)

    def forward(self, h, mask):
        maskf = mask.unsqueeze(-1).float()
        mean = (h * maskf).sum(1) / torch.clamp(maskf.sum(1), min=1.0)

        scores = self.attn(h).squeeze(-1)
        scores = scores.masked_fill(mask == 0, -1e4)
        w = torch.softmax(scores, dim=1).unsqueeze(-1)
        attn = (h * w).sum(1)
        return torch.cat([mean, attn], dim=1)  # (B, 2H)

class MLMHybridMultiLabel(nn.Module):
    def __init__(self, encoder, bb_vocab_size, fp_dim, n_targets):
        super().__init__()
        self.encoder = encoder
        H = self.encoder.config.hidden_size
        self.pool = MeanAttnPool(H)

        self.bb_emb = nn.Embedding(bb_vocab_size, 64, padding_idx=bb_pad)
        self.bb_proj = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.1))

        self.fp_proj = nn.Sequential(
            nn.Linear(fp_dim, 512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.1)
        )

        fusion_in = 2*H + 128 + 256
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_in),
            nn.Dropout(0.2),
            nn.Linear(fusion_in, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, n_targets)
        )

    def forward(self, input_ids, attention_mask, fp_bits, bb_ids, use_bb: bool):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        pooled = self.pool(h, attention_mask)  # (B, 2H)

        if use_bb and (bb_ids is not None):
            bb = self.bb_emb(bb_ids).mean(1)
            bb = self.bb_proj(bb)
        else:
            bb = pooled.new_zeros((pooled.size(0), 128))

        fp = self.fp_proj(fp_bits)  # fp_bits already zeroed when ablated

        fused = torch.cat([pooled, bb, fp], dim=1)
        logits = self.head(fused)
        return logits


In [10]:
def make_pos_weight(y: np.ndarray) -> torch.Tensor:
    pos = y.sum(0).astype(np.float32)
    neg = (len(y) - pos).astype(np.float32)
    w = np.clip(neg / np.clip(pos, 1.0, None), 1.0, 100.0)
    return torch.tensor(w, dtype=torch.float32, device=device)

def build_loss_fn(y_train: np.ndarray, use_pos_weight: bool, lambda_prom: float):
    if use_pos_weight:
        pos_weight = make_pos_weight(y_train)
        bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        bce = nn.BCEWithLogitsLoss()

    def loss_fn(logits, targets):
        base = bce(logits, targets)
        if lambda_prom <= 0:
            return base
        p = torch.sigmoid(logits)
        reg = ((p.sum(1) - targets.sum(1))**2).mean()
        return base + lambda_prom * reg
    return loss_fn

def sigmoid_np(x): return 1.0 / (1.0 + np.exp(-x))

def ece_multilabel(probs, y_true, n_bins=15):
    bins = np.linspace(0, 1, n_bins+1)
    eces = []
    for j in range(y_true.shape[1]):
        p = probs[:, j]; t = y_true[:, j]
        ece = 0.0
        for i in range(n_bins):
            m = (p >= bins[i]) & (p < bins[i+1])
            if m.sum() == 0:
                continue
            acc = t[m].mean()
            conf = p[m].mean()
            ece += (m.mean()) * abs(acc - conf)
        eces.append(ece)
    return float(np.mean(eces))

def brier_mean(probs, y_true):
    return float(np.mean((probs - y_true)**2))

def ap_table(y_true, y_prob) -> Dict[str, float]:
    out = {
        "AP_micro": float(average_precision_score(y_true, y_prob, average="micro")),
        "AP_macro": float(average_precision_score(y_true, y_prob, average="macro")),
    }
    for k, t in enumerate(cfg.targets):
        out[f"AP_{t}"] = float(average_precision_score(y_true[:, k], y_prob[:, k]))
    return out


In [11]:
import inspect

def mlm_ckpt_path():
    return os.path.join(cfg.out_dir, cfg.mlm_out_name)

def mlm_ready(path: str) -> bool:
    return os.path.exists(os.path.join(path, "config.json")) and (
        os.path.exists(os.path.join(path, "pytorch_model.bin")) or
        os.path.exists(os.path.join(path, "model.safetensors"))
    )

def build_training_args(**kwargs):
    sig = inspect.signature(TrainingArguments.__init__).parameters
    # Transformers changed naming in some versions; support both
    if "evaluation_strategy" in sig and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    if "save_strategy" not in sig and "save_strategy" in kwargs:
        kwargs.pop("save_strategy")
    return TrainingArguments(**kwargs)

def run_mlm_dapt():
    out = mlm_ckpt_path()
    os.makedirs(out, exist_ok=True)

    train_smiles = train_df[cfg.smiles_col].astype(str).values
    if len(train_smiles) > cfg.mlm_sample_max:
        rs = np.random.RandomState(42)
        train_smiles = train_smiles[rs.choice(len(train_smiles), cfg.mlm_sample_max, replace=False)]

    val_smiles = val_df[cfg.smiles_col].astype(str).values[:cfg.mlm_val_max]

    mlm_tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)

    class SmilesMLMDataset(Dataset):
        def __init__(self, smiles_list):
            self.smiles = [str(s) for s in smiles_list]
        def __len__(self): return len(self.smiles)
        def __getitem__(self, idx):
            return mlm_tokenizer(
                self.smiles[idx],
                truncation=True,
                max_length=cfg.max_len,
                return_special_tokens_mask=True
            )

    ds_tr = SmilesMLMDataset(train_smiles)
    ds_va = SmilesMLMDataset(val_smiles)

    collator = DataCollatorForLanguageModeling(
        tokenizer=mlm_tokenizer, mlm=True, mlm_probability=cfg.mlm_prob
    )

    model = AutoModelForMaskedLM.from_pretrained(cfg.base_model)

    args = build_training_args(
        output_dir=out,
        overwrite_output_dir=True,
        per_device_train_batch_size=cfg.mlm_batch,
        per_device_eval_batch_size=cfg.mlm_batch,
        max_steps=cfg.mlm_max_steps,
        learning_rate=cfg.mlm_lr,
        weight_decay=cfg.weight_decay,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=max(200, cfg.mlm_max_steps // 4),
        logging_steps=50,
        load_best_model_at_end=False,
        fp16=(cfg.use_fp16 and torch.cuda.is_available()),
        report_to="none",
        dataloader_num_workers=0,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        data_collator=collator,
        tokenizer=mlm_tokenizer,
    )
    trainer.train()
    trainer.save_model(out)
    mlm_tokenizer.save_pretrained(out)
    print("[mlm] saved:", out)

if cfg.do_mlm:
    if not mlm_ready(mlm_ckpt_path()):
        print("[mlm] running DAPT…")
        run_mlm_dapt()
    else:
        print("[mlm] checkpoint ready:", mlm_ckpt_path())


[mlm] running DAPT…


Some weights of the model checkpoint at seyonec/ChemBERTa-zinc-base-v1 were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss,Validation Loss
50,0.977300,0.477230
100,0.483600,0.390440
150,0.413600,0.347354
200,0.359600,0.328213
250,0.361400,0.303995
300,0.335700,0.294665
350,0.311000,0.278776
400,0.307100,0.273954
450,0.300900,0.265417
500,0.302400,0.255170


[mlm] saved: ./ablation_q1_aligned\ckpt_mlm_belka


In [12]:
def freeze_all_encoder(m):
    for p in m.encoder.parameters():
        p.requires_grad = False

def unfreeze_last_k_layers(m, k=2):
    freeze_all_encoder(m)
    candidates = []
    if hasattr(m.encoder, "encoder") and hasattr(m.encoder.encoder, "layer"):
        candidates = list(m.encoder.encoder.layer)
    elif hasattr(m.encoder, "roberta") and hasattr(m.encoder.roberta, "encoder") and hasattr(m.encoder.roberta.encoder, "layer"):
        candidates = list(m.encoder.roberta.encoder.layer)
    if candidates:
        for layer in candidates[-k:]:
            for p in layer.parameters():
                p.requires_grad = True
    else:
        # fallback
        for p in m.encoder.parameters():
            p.requires_grad = True

@torch.no_grad()
def predict_logits(model, dl, use_bb: bool):
    model.eval()
    logits_all, y_all = [], []
    for batch in tqdm(dl, leave=False):
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        fp = batch["fp_bits"].to(device)
        bb = None if batch["bb_ids"] is None else batch["bb_ids"].to(device)
        logits = model(input_ids, attn, fp, bb, use_bb=use_bb)
        logits_all.append(logits.float().cpu().numpy())
        if "labels" in batch:
            y_all.append(batch["labels"].cpu().numpy())
    return np.concatenate(logits_all, 0), (np.concatenate(y_all, 0) if y_all else None)

def train_phase(
    model, dl_tr, dl_va, loss_fn,
    epochs: int, lr_head: float, lr_enc: float, unfreeze_k: int,
    use_bb: bool, max_steps: Optional[int], phase_name: str
):
    if unfreeze_k == 0:
        freeze_all_encoder(model)
    else:
        unfreeze_last_k_layers(model, k=unfreeze_k)

    head_params, enc_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("encoder."):
            enc_params.append(p)
        else:
            head_params.append(p)

    opt_groups = []
    if head_params:
        opt_groups.append({"params": head_params, "lr": lr_head})
    if enc_params:
        opt_groups.append({"params": enc_params, "lr": lr_enc})

    opt = torch.optim.AdamW(opt_groups, weight_decay=cfg.weight_decay)

    # total optimizer steps (no grad accumulation here; add if you need)
    total_steps = len(dl_tr) * epochs
    if max_steps is not None:
        total_steps = min(total_steps, max_steps)
    warmup = int(cfg.warmup_frac * total_steps)
    sched = get_cosine_schedule_with_warmup(opt, warmup, total_steps)

    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_fp16 and device.type == "cuda"))

    best_ap = -1.0
    best_state = None
    step = 0

    print(f"\n=== {phase_name} ===")
    for ep in range(1, epochs + 1):
        model.train()
        losses = []

        for batch in tqdm(dl_tr, desc=f"{phase_name} ep{ep}/{epochs}", leave=False):
            opt.zero_grad(set_to_none=True)
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            fp = batch["fp_bits"].to(device)
            bb = None if batch["bb_ids"] is None else batch["bb_ids"].to(device)
            y = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=(cfg.use_fp16 and device.type == "cuda")):
                logits = model(input_ids, attn, fp, bb, use_bb=use_bb)
                loss = loss_fn(logits, y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()

            losses.append(float(loss.detach().cpu()))
            step += 1
            if max_steps is not None and step >= max_steps:
                break

        # Val micro-AP for checkpoint select (Hybrid-Final contract)
        va_logits, va_y = predict_logits(model, dl_va, use_bb=use_bb)
        va_prob = sigmoid_np(va_logits)
        va_micro = average_precision_score(va_y, va_prob, average="micro")
        print(f"{phase_name} ep{ep}: train_loss={np.mean(losses):.4f} | val micro-AP={va_micro:.4f}")

        if va_micro > best_ap:
            best_ap = va_micro
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if max_steps is not None and step >= max_steps:
            break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)
    print(f"{phase_name} best val micro-AP: {best_ap:.4f}")
    return best_ap


In [13]:
def nll_from_logits(logits_1d, y_1d, T):
    z = logits_1d / T
    p = 1.0 / (1.0 + np.exp(-z))
    eps = 1e-9
    return -np.mean(y_1d*np.log(p+eps) + (1-y_1d)*np.log(1-p+eps))

def fit_temps(va_logits, y_va):
    T_grid = np.logspace(-1, 1, 41)  # 0.1..10
    temps = np.ones(y_va.shape[1], dtype=np.float32)
    for j in range(y_va.shape[1]):
        best = (1e18, 1.0)
        for T in T_grid:
            loss = nll_from_logits(va_logits[:, j], y_va[:, j], T)
            if loss < best[0]:
                best = (loss, T)
        temps[j] = best[1]
    return temps

def apply_temp(logits, temps):
    out = []
    for j in range(logits.shape[1]):
        z = logits[:, j] / temps[j]
        out.append(1.0 / (1.0 + np.exp(-z)))
    return np.stack(out, axis=1).astype(np.float32)

def find_threshold_for_precision(y_true, y_prob, target_precision=0.9):
    thr_grid = np.linspace(0.99, 0.01, 99)
    best = None
    for thr in thr_grid:
        pred = (y_prob >= thr).astype(int)
        if pred.sum() == 0:
            continue
        prec = precision_score(y_true, pred, zero_division=0)
        rec = recall_score(y_true, pred, zero_division=0)
        if prec >= target_precision:
            if best is None or rec > best["rec"]:
                best = {"thr": thr, "prec": prec, "rec": rec}
    return 0.5 if best is None else float(best["thr"])

def safe_thresholds(y_va, va_prob_cal):
    thr = np.zeros(y_va.shape[1], dtype=np.float32)
    for j in range(y_va.shape[1]):
        thr[j] = find_threshold_for_precision(y_va[:, j], va_prob_cal[:, j], cfg.target_precision)
    return thr

def safe_decode(prob, thresholds):
    return (prob >= thresholds.reshape(1, -1)).astype(int)

def hallucination_counts(y_true, y_pred):
    gt_cnt = y_true.sum(1)
    pr_cnt = y_pred.sum(1)
    return {
        "pred_triple": int((pr_cnt == 3).sum()),
        "triple_when_gt<=1": int(((gt_cnt <= 1) & (pr_cnt == 3)).sum()),
        "pred_zero": int((pr_cnt == 0).sum()),
    }


In [14]:
def run_variant_once(spec: dict, seed: int):
    seed_everything(seed)

    # Encoder choice
    if spec["encoder"] == "base":
        enc_name = cfg.base_model
    elif spec["encoder"] == "mlm":
        enc_name = mlm_ckpt_path()
    else:
        raise ValueError("encoder must be base|mlm")

    tokenizer = AutoTokenizer.from_pretrained(enc_name, use_fast=True)
    encoder = AutoModel.from_pretrained(enc_name)

    model = MLMHybridMultiLabel(
        encoder=encoder,
        bb_vocab_size=len(bb_vocab),
        fp_dim=cfg.fp_bits,
        n_targets=len(cfg.targets)
    ).to(device)

    use_fp = spec["use_fp"]
    use_bb = spec["use_bb"]
    do_ft  = spec["do_ft"]

    # loaders (full val/test; optional random eval cap only if cfg.eval_max_n set)
    ds_tr = BelkaDataset(train_df, fp_tr, bb_tr, y_tr)
    ds_va = BelkaDataset(maybe_cap_eval(val_df),  fp_va, bb_va, y_va)
    ds_te = BelkaDataset(maybe_cap_eval(test_df), fp_te, bb_te, y_te)

    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True, num_workers=0,
                       collate_fn=make_collate(tokenizer, use_fp=use_fp, use_bb=use_bb))
    dl_va = DataLoader(ds_va, batch_size=cfg.batch_size*cfg.eval_batch_mult, shuffle=False, num_workers=0,
                       collate_fn=make_collate(tokenizer, use_fp=use_fp, use_bb=use_bb))
    dl_te = DataLoader(ds_te, batch_size=cfg.batch_size*cfg.eval_batch_mult, shuffle=False, num_workers=0,
                       collate_fn=make_collate(tokenizer, use_fp=use_fp, use_bb=use_bb))

    # Loss (faithful unless explicitly ablated)
    loss_fn = build_loss_fn(
        y_train=y_tr,
        use_pos_weight=spec["use_pos_weight"],
        lambda_prom=spec["lambda_promiscuity"]
    )

    t0 = time.time()

    # Phase A: head-only
    train_phase(
        model, dl_tr, dl_va, loss_fn,
        epochs=cfg.epochs_head, lr_head=cfg.lr_head, lr_enc=0.0, unfreeze_k=0,
        use_bb=use_bb, max_steps=cfg.max_steps_head,
        phase_name=f"{spec['name']} [seed{seed}] PhaseA"
    )

    # Phase B: finetune last-k
    if do_ft:
        train_phase(
            model, dl_tr, dl_va, loss_fn,
            epochs=cfg.epochs_finetune, lr_head=cfg.lr_head/3, lr_enc=cfg.lr_encoder, unfreeze_k=cfg.unfreeze_k,
            use_bb=use_bb, max_steps=cfg.max_steps_ft,
            phase_name=f"{spec['name']} [seed{seed}] PhaseB"
        )

    # VAL logits for temperature scaling + threshold selection
    va_logits, va_y_float = predict_logits(model, dl_va, use_bb=use_bb)
    va_y_int = va_y_float.astype(int)

    temps = fit_temps(va_logits, va_y_int)
    va_prob_cal = apply_temp(va_logits, temps)
    thresholds = safe_thresholds(va_y_int, va_prob_cal)

    # TEST evaluation (same test set for ALL variants!)
    te_logits, te_y_float = predict_logits(model, dl_te, use_bb=use_bb)
    te_y_int = te_y_float.astype(int)
    te_prob_cal = apply_temp(te_logits, temps)

    # Metrics
    metrics = {}
    metrics |= {f"VAL_{k}": v for k, v in ap_table(va_y_int, va_prob_cal).items()}
    metrics |= {f"TEST_{k}": v for k, v in ap_table(te_y_int, te_prob_cal).items()}
    metrics["TEST_ECE"] = ece_multilabel(te_prob_cal, te_y_int)
    metrics["TEST_Brier"] = brier_mean(te_prob_cal, te_y_int)

    # Safe decoding metrics
    te_pred_safe = safe_decode(te_prob_cal, thresholds)
    metrics["SAFE_halluc_pred_triple"] = hallucination_counts(te_y_int, te_pred_safe)["pred_triple"]
    metrics["SAFE_halluc_triple_when_gt<=1"] = hallucination_counts(te_y_int, te_pred_safe)["triple_when_gt<=1"]

    for j, t in enumerate(cfg.targets):
        metrics[f"SAFE_thr_{t}"] = float(thresholds[j])
        metrics[f"SAFE_prec_{t}"] = float(precision_score(te_y_int[:, j], te_pred_safe[:, j], zero_division=0))
        metrics[f"SAFE_rec_{t}"]  = float(recall_score(te_y_int[:, j], te_pred_safe[:, j], zero_division=0))
        metrics[f"SAFE_f1_{t}"]   = float(f1_score(te_y_int[:, j], te_pred_safe[:, j], zero_division=0))

    row = {
        "variant": spec["name"],
        "seed": seed,
        "encoder": spec["encoder"],
        "use_fp": use_fp,
        "use_bb": use_bb,
        "do_ft": do_ft,
        "use_pos_weight": spec["use_pos_weight"],
        "lambda_promiscuity": spec["lambda_promiscuity"],
        "elapsed_sec": round(time.time() - t0, 2),
    }
    row.update(metrics)

    # cleanup
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row


In [15]:
ABLATIONS = [
    # Full method (reference)
    dict(name="FULL_MLM_FP_BB_FT", encoder="mlm", use_fp=True,  use_bb=True,  do_ft=True,
         use_pos_weight=True,  lambda_promiscuity=cfg.lambda_promiscuity),

    # One-factor removals (clean ablations)
    dict(name="w/o_DAPT",          encoder="base", use_fp=True,  use_bb=True,  do_ft=True,
         use_pos_weight=True,  lambda_promiscuity=cfg.lambda_promiscuity),

    dict(name="w/o_FP",            encoder="mlm",  use_fp=False, use_bb=True,  do_ft=True,
         use_pos_weight=True,  lambda_promiscuity=cfg.lambda_promiscuity),

    dict(name="w/o_BB",            encoder="mlm",  use_fp=True,  use_bb=False, do_ft=True,
         use_pos_weight=True,  lambda_promiscuity=cfg.lambda_promiscuity),

    dict(name="w/o_FT",            encoder="mlm",  use_fp=True,  use_bb=True,  do_ft=False,
         use_pos_weight=True,  lambda_promiscuity=cfg.lambda_promiscuity),

    dict(name="w/o_pos_weight",    encoder="mlm",  use_fp=True,  use_bb=True,  do_ft=True,
         use_pos_weight=False, lambda_promiscuity=cfg.lambda_promiscuity),

    dict(name="w/o_promisc_reg",   encoder="mlm",  use_fp=True,  use_bb=True,  do_ft=True,
         use_pos_weight=True,  lambda_promiscuity=0.0),

    # Optional ladder endpoint (SMILES-only)
    dict(name="SMILES_only",       encoder="mlm",  use_fp=False, use_bb=False, do_ft=True,
         use_pos_weight=True,  lambda_promiscuity=cfg.lambda_promiscuity),
]

results_jsonl = os.path.join(cfg.out_dir, "ablation_results.jsonl")
results_csv   = os.path.join(cfg.out_dir, "ablation_results.csv")

def load_done():
    if not os.path.exists(results_jsonl):
        return set()
    done = set()
    with open(results_jsonl, "r") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                done.add((r["variant"], r["seed"]))
    return done

done = load_done()
print("already done:", len(done))

all_rows = []
for seed in cfg.seeds:
    for spec in ABLATIONS:
        key = (spec["name"], seed)
        if key in done:
            print("skip:", key)
            continue
        row = run_variant_once(spec, seed)
        all_rows.append(row)
        with open(results_jsonl, "a") as f:
            f.write(json.dumps(row) + "\n")
        print("saved:", key, "| TEST_AP_micro:", row["TEST_AP_micro"])

# Export CSV snapshot
rows = []
with open(results_jsonl, "r") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
res = pd.DataFrame(rows)
res.to_csv(results_csv, index=False)
print("saved:", results_csv)
res.head()


already done: 0


Some weights of RobertaModel were not initialized from the model checkpoint at ./ablation_q1_aligned\ckpt_mlm_belka and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== FULL_MLM_FP_BB_FT [seed42] PhaseA ===


FULL_MLM_FP_BB_FT [seed42] PhaseA ep1/1:   0%|          | 0/32196 [00:00<?, ?it/s]

  0%|          | 0/4025 [00:00<?, ?it/s]

...

SMILES_only [seed42] PhaseB ep1: train_loss=0.3745 | val micro-AP=0.9341
SMILES_only [seed42] PhaseB best val micro-AP: 0.9341


  0%|          | 0/4025 [00:00<?, ?it/s]

  0%|          | 0/5031 [00:00<?, ?it/s]

saved: ('SMILES_only', 42) | TEST_AP_micro: 0.9343733188429184
saved: ./ablation_q1_aligned\ablation_results.csv


,variant,seed,encoder,use_fp,use_bb,do_ft,use_pos_weight,lambda_promiscuity,elapsed_sec,VAL_AP_micro,...,SAFE_rec_BRD4,SAFE_f1_BRD4,SAFE_thr_HSA,SAFE_prec_HSA,SAFE_rec_HSA,SAFE_f1_HSA,SAFE_thr_sEH,SAFE_prec_sEH,SAFE_rec_sEH,SAFE_f1_sEH
0,FULL_MLM_FP_BB_FT,42,mlm,True,True,True,True,0.05,2112.25,0.953282,...,0.947817,0.923587,0.87,0.907896,0.274665,0.421742,0.60,0.902434,0.890952,0.896656
1,w/o_DAPT,42,base,True,True,True,True,0.05,2036.93,0.953051,...,0.948375,0.923658,0.86,0.907291,0.279632,0.427504,0.59,0.904536,0.885871,0.895106
2,w/o_FP,42,mlm,False,True,True,True,0.05,2040.16,0.946099,...,0.921955,0.911485,0.88,0.902386,0.235426,0.373427,0.63,0.903002,0.869807,0.886094
3,w/o_BB,42,mlm,True,False,True,True,0.05,2038.25,0.953093,...,0.944260,0.921980,0.87,0.905732,0.288757,0.437905,0.61,0.903032,0.892022,0.897493
4,w/o_FT,42,mlm,True,True,False,True,0.05,1526.31,0.949906,...,0.935230,0.918241,0.88,0.904786,0.263155,0.407725,0.60,0.903960,0.875516,0.889511


In [16]:
res = pd.read_csv(results_csv)

key_cols = ["variant"]
metric_cols = [c for c in res.columns if c.startswith("TEST_AP_") or c in ["TEST_ECE","TEST_Brier","elapsed_sec"]]

agg = (
    res.groupby(key_cols)[metric_cols]
      .agg(["mean","std"])
      .reset_index()
)

# Delta vs baseline (use mean)
baseline = "FULL_MLM_FP_BB_FT"
base_mean = res[res["variant"] == baseline].groupby("variant")[metric_cols].mean().iloc[0]

delta_rows = []
for v in res["variant"].unique():
    v_mean = res[res["variant"] == v].groupby("variant")[metric_cols].mean().iloc[0]
    d = (v_mean - base_mean).to_dict()
    d["variant"] = v
    delta_rows.append(d)

delta = pd.DataFrame(delta_rows).sort_values("TEST_AP_micro", ascending=False)

print("=== Mean±Std (by variant) ===")
display(agg)

print("\n=== Delta vs FULL (means) ===")
display(delta[["variant","TEST_AP_micro","TEST_AP_macro","TEST_ECE","TEST_Brier","elapsed_sec"]])


=== Mean±Std (by variant) ===


variant elapsed_sec     TEST_AP_micro     TEST_AP_macro      \
                            mean std          mean std          mean std   
0  FULL_MLM_FP_BB_FT     2112.25 NaN      0.953499 NaN      0.928970 NaN   
1        SMILES_only     2038.38 NaN      0.934373 NaN      0.906258 NaN   
2             w/o_BB     2038.25 NaN      0.953329 NaN      0.929027 NaN   
3           w/o_DAPT     2036.93 NaN      0.953117 NaN      0.929257 NaN   
4             w/o_FP     2040.16 NaN      0.946021 NaN      0.919508 NaN   
5             w/o_FT     1526.31 NaN      0.949958 NaN      0.924799 NaN   
6     w/o_pos_weight     2039.28 NaN      0.956817 NaN      0.930584 NaN   
7    w/o_promisc_reg     2039.99 NaN      0.953278 NaN      0.928910 NaN   

  TEST_AP_BRD4     TEST_AP_HSA     TEST_AP_sEH      TEST_ECE     TEST_Brier  \
          mean std        mean std        mean std      mean std       mean   
0     0.978591 NaN    0.834348 NaN    0.973972 NaN  0.036646 NaN   0.061995   
1     0.953588 NaN    0.801166 NaN    0.964021 NaN  0.049735 NaN   0.077118   
2     0.977520 NaN    0.835423 NaN    0.974138 NaN  0.037605 NaN   0.062594   
3     0.978556 NaN    0.835479 NaN    0.973734 NaN  0.035965 NaN   0.062437   
4     0.971368 NaN    0.818572 NaN    0.968583 NaN  0.041791 NaN   0.068424   
5     0.975271 NaN    0.827480 NaN    0.971646 NaN  0.038358 NaN   0.065003   
6     0.979314 NaN    0.837054 NaN    0.975385 NaN  0.005431 NaN   0.056069   
7     0.978335 NaN    0.834667 NaN    0.973729 NaN  0.039626 NaN   0.062883   

       
  std  
0 NaN  
1 NaN  
2 NaN  
3 NaN  
4 NaN  
5 NaN  
6 NaN  
7 NaN


=== Delta vs FULL (means) ===


,variant,TEST_AP_micro,TEST_AP_macro,TEST_ECE,TEST_Brier,elapsed_sec
5,w/o_pos_weight,0.003318,0.001614,-0.031215,-0.005926,-72.97
0,FULL_MLM_FP_BB_FT,0.000000,0.000000,0.000000,0.000000,0.00
3,w/o_BB,-0.000170,0.000057,0.000959,0.000599,-74.00
6,w/o_promisc_reg,-0.000221,-0.000060,0.002980,0.000888,-72.26
1,w/o_DAPT,-0.000381,0.000287,-0.000681,0.000442,-75.32
4,w/o_FT,-0.003541,-0.004171,0.001712,0.003008,-585.94
2,w/o_FP,-0.007478,-0.009462,0.005145,0.006429,-72.09
7,SMILES_only,-0.019126,-0.022712,0.013089,0.015123,-73.87


In [21]:
res = pd.read_csv(results_csv)

key_cols = ["variant"]
metric_cols = [c for c in res.columns if c.startswith("TEST_AP_") or c in ["TEST_ECE","TEST_Brier","elapsed_sec"]]

# Mean±Std (std uses ddof=0 so single-run -> 0 instead of NaN)
agg = (
    res.groupby(key_cols)[metric_cols]
      .agg([("mean","mean"), ("std", lambda x: x.std(ddof=0))])
      .reset_index()
)

# Sort Mean table by TEST_AP_micro mean (desc)
agg_sorted = agg.sort_values(("TEST_AP_micro", "mean"), ascending=False)

# ---- Delta vs SMILES_only baseline ----
baseline = "SMILES_only"
if baseline not in set(res["variant"]):
    raise ValueError(f"Baseline '{baseline}' not found in res['variant'].")

base_mean = (
    res[res["variant"] == baseline]
    .groupby("variant")[metric_cols]
    .mean()
    .iloc[0]
)

delta_rows = []
for v in res["variant"].unique():
    v_mean = (
        res[res["variant"] == v]
        .groupby("variant")[metric_cols]
        .mean()
        .iloc[0]
    )
    d = (v_mean - base_mean).to_dict()
    d["variant"] = v
    delta_rows.append(d)

delta = pd.DataFrame(delta_rows)

# Keep sorting like before: sort ALL rows by TEST_AP_micro delta (desc)
delta_sorted = delta.sort_values("TEST_AP_micro", ascending=False)

print("=== Mean±Std (by variant), sorted by TEST_AP_micro mean (desc) ===")
display(agg_sorted)

print(f"\n=== Delta vs {baseline} (means), sorted by TEST_AP_micro delta (desc) ===")
display(delta_sorted[["variant","TEST_AP_micro","TEST_AP_macro","TEST_ECE","TEST_Brier","elapsed_sec"]])


=== Mean±Std (by variant), sorted by TEST_AP_micro mean (desc) ===


variant elapsed_sec      TEST_AP_micro      TEST_AP_macro       \
                            mean  std          mean  std          mean  std   
6     w/o_pos_weight     2039.28  0.0      0.956817  0.0      0.930584  0.0   
0  FULL_MLM_FP_BB_FT     2112.25  0.0      0.953499  0.0      0.928970  0.0   
2             w/o_BB     2038.25  0.0      0.953329  0.0      0.929027  0.0   
7    w/o_promisc_reg     2039.99  0.0      0.953278  0.0      0.928910  0.0   
3           w/o_DAPT     2036.93  0.0      0.953117  0.0      0.929257  0.0   
5             w/o_FT     1526.31  0.0      0.949958  0.0      0.924799  0.0   
4             w/o_FP     2040.16  0.0      0.946021  0.0      0.919508  0.0   
1        SMILES_only     2038.38  0.0      0.934373  0.0      0.906258  0.0   

  TEST_AP_BRD4      TEST_AP_HSA      TEST_AP_sEH       TEST_ECE       \
          mean  std        mean  std        mean  std      mean  std   
6     0.979314  0.0    0.837054  0.0    0.975385  0.0  0.005431  0.0   
0     0.978591  0.0    0.834348  0.0    0.973972  0.0  0.036646  0.0   
2     0.977520  0.0    0.835423  0.0    0.974138  0.0  0.037605  0.0   
7     0.978335  0.0    0.834667  0.0    0.973729  0.0  0.039626  0.0   
3     0.978556  0.0    0.835479  0.0    0.973734  0.0  0.035965  0.0   
5     0.975271  0.0    0.827480  0.0    0.971646  0.0  0.038358  0.0   
4     0.971368  0.0    0.818572  0.0    0.968583  0.0  0.041791  0.0   
1     0.953588  0.0    0.801166  0.0    0.964021  0.0  0.049735  0.0   

  TEST_Brier       
        mean  std  
6   0.056069  0.0  
0   0.061995  0.0  
2   0.062594  0.0  
7   0.062883  0.0  
3   0.062437  0.0  
5   0.065003  0.0  
4   0.068424  0.0  
1   0.077118  0.0


=== Delta vs SMILES_only (means), sorted by TEST_AP_micro delta (desc) ===


,variant,TEST_AP_micro,TEST_AP_macro,TEST_ECE,TEST_Brier,elapsed_sec
5,w/o_pos_weight,0.022444,0.024326,-0.044304,-0.021050,0.90
0,FULL_MLM_FP_BB_FT,0.019126,0.022712,-0.013089,-0.015123,73.87
3,w/o_BB,0.018956,0.022769,-0.012129,-0.014524,-0.13
6,w/o_promisc_reg,0.018905,0.022652,-0.010109,-0.014235,1.61
1,w/o_DAPT,0.018744,0.022998,-0.013770,-0.014681,-1.45
4,w/o_FT,0.015585,0.018541,-0.011377,-0.012115,-512.07
2,w/o_FP,0.011648,0.013250,-0.007943,-0.008694,1.78
7,SMILES_only,0.000000,0.000000,0.000000,0.000000,0.00
